# Setup

In [1]:
import json, os

with open('/home/ubuntu/cabir/keys/6.4.1.spark_nlp_for_healthcare.json') as f:
    license_keys = json.load(f)

# Defining license key-value pairs as local variables
locals().update(license_keys)
os.environ.update(license_keys)

In [ ]:
# Installing pyspark and spark-nlp
! pip install --upgrade -q pyspark==3.5.1 spark-nlp==$PUBLIC_VERSION

# Installing Spark NLP Healthcare
! pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION  --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Installing Spark NLP Display Library for visualization
! pip install -q spark-nlp-display

In [ ]:
import json
import os

import sparknlp
import sparknlp_jsl

from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp_jsl.pipeline_tracer import PipelineTracer
from sparknlp_jsl.pipeline_output_parser import PipelineOutputParser

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline,PipelineModel


import pandas as pd
pd.set_option('display.max_colwidth', 200)

import warnings
warnings.filterwarnings('ignore')

spark = sparknlp_jsl.start(license_keys['SECRET'],
                           #params=params
                           )

spark.sparkContext.setLogLevel("ERROR")

print("Spark NLP Version :", sparknlp.version())
print("Spark NLP_JSL Version :", sparknlp_jsl.version())

spark


Invalid maximum heap size: -Xmx0G
Error: Could not create the Java Virtual Machine.
Error: A fatal exception has occurred. Program will exit.


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

# PretrainedZeroShotMultiTask — end-to-end multi-task extraction

A single zero-shot model runs **four tasks in one pass**: NER, classification, structured (JSON)
extraction, and relation extraction.

**Flow:** read config → normalize the 4 schemas into the annotator's syntax → build pipeline → run → flattened JSON.

**Annotator syntax (what we normalize into):**

| Task | Setter | Shape |
|------|--------|-------|
| Entities | `setEntities` | `["LABEL::dtype::description"]` |
| Structures | `setStructures` | `[("name", ["field::dtype::desc", "field::[a\|b\|c]"])]` |
| Classifications | `setClassifications` | `[("task", ["l1", "l2"])]` |
| Relations | `setRelations` | `["subject_verb_object"]` |

In [ ]:
# Base multi-task model (~805 MB). For harder clinical text, swap to the more capable
# "zeroshot_multitask_generic" — same API, just a different resource name.
zeroshot_multitask = PretrainedZeroShotMultiTask\
                            .pretrained("zeroshot_multitask_base", "en", "clinical/models")

zeroshot_multitask_base download started this may take some time.
Approximate size to download 805.4 MB
[ | ]zeroshot_multitask_base download started this may take some time.
Approximate size to download 805.4 MB
Download done! Loading the resource.
[ / ]

[ | ]

[OK!]


## 1. Config

The rich input config our services receive. `route` uses a `choices` list (→ enum field).

In [5]:
zeroshot_input_prompt_dict = {
    "entities": [
        {"label": "PROBLEM",    "dtype": "str", "description": "A medical condition, symptom, or diagnosis"},
        {"label": "MEDICATION", "dtype": "str", "description": "Drug or pharmaceutical treatment"},
        {"label": "PROCEDURE",  "dtype": "str", "description": "Medical or surgical procedure"},
        {"label": "TEST",       "dtype": "str", "description": "Diagnostic test or lab result"},
    ],
    "structures": [
        {"name": "medication_item", "fields": [
            {"name": "drug_name", "dtype": "str", "description": "Name of the drug"},
            {"name": "dosage",    "dtype": "str", "description": "Dose amount and unit"},
            {"name": "frequency", "dtype": "str", "description": "How often taken"},
            {"name": "route",     "choices": ["oral", "IV", "topical", "subcutaneous"]},  # enum field
        ]},
    ],
    "classifications": [
        {"task": "document_type", "labels": ["Radiology Report", "Discharge Summary", "Progress Note"]},
    ],
    "relations": ["MEDICATION treats PROBLEM", "TEST diagnoses PROBLEM"],
    "entity_threshold": 0.6, 
    "structure_threshold": 0.6,
    "classification_threshold": 0.6, 
    "relation_threshold": 0.6,
}

## 2. Normalize → annotator syntax

Flatten each schema into the shapes the annotator expects.

In [ ]:
# These normalizers used to be defined here and hand-copied into custom_nlp_service, with a note
# to import a shared module "once one exists". It exists now: nlp-services-standalone/app/
# zeroshot_dsl.py, which the standalone service imports too -- so the notebook and the service
# cannot drift apart. It is stdlib-only, so importing it here pulls in nothing else.
import sys

sys.path.insert(0, "/home/ubuntu/cabir/multitask-gliner2/multitask-container/nlp-services-standalone/app")

from zeroshot_dsl import normalize_config, validate_annotator_config

In [ ]:
# The assert block that used to live here is now zeroshot_dsl.validate_annotator_config(), which
# the service's request model calls as well -- so a config this notebook accepts is exactly a
# config the service accepts (it checks the key set matches build_multitask_pipeline()'s
# parameters, and that relation names are single tokens).
zeroshot_input_prompt_dict = validate_annotator_config(normalize_config(zeroshot_input_prompt_dict))

print("[PASS] normalize_config() produced valid annotator syntax:")
for k in ("entities", "structures", "classifications", "relations"):
    print(f"  {k:16} {zeroshot_input_prompt_dict[k]}")

# `zeroshot_input_prompt_dict` is exactly what build_multitask_pipeline(**...) takes below.

## 3. Build pipeline

In [16]:
# The annotator consumes the raw `document` directly. The sentence_detector / tokenizer /
# document_splitter stages are only for windowing very long docs into `splits`; not used by default.
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

sentence_detector = SentenceDetectorDLModel\
    .pretrained("sentence_detector_dl_healthcare_v2_wip","en","clinical/models")\
    .setInputCols(["document"])\
    .setOutputCol("sentence")

# sentence_detector = SentenceDetector()\
#     .setInputCols("document")\
#     .setOutputCol("sentence")

# tokenizer = Tokenizer()\
#     .setInputCols("document")\
#     .setOutputCol("token")

# document_splitter = InternalDocumentSplitter()\
#     .setInputCols("document", "sentence", "token")\
#     .setOutputCol("splits")\
#     .setSplitMode("token")\
#     .setSentenceAwareness(True)\
#     .setMaxLength(120)

sentence_detector_dl_healthcare_v2_wip download started this may take some time.
Approximate size to download 368.6 KB
[ | ]

[OK!]


In [17]:
def build_multitask_pipeline(
    entities=None, structures=None, classifications=None, relations=None,
    entity_threshold=0.6, structure_threshold=0.6,
    classification_threshold=0.6, relation_threshold=0.6,
):
    """Build a fitted PipelineModel wiring all four zero-shot tasks.

    Each argument is already in the annotator's flattened syntax (see normalize_config):
      entities         -> ["LABEL::dtype::description", ...]
      structures       -> [("struct_name", ["field::dtype::description", "field::[a|b|c]"]), ...]
      classifications  -> [("task_name", ["label1", "label2"]), ...]
      relations        -> ["subject_verb_object", ...]
    """
    zero_shot = (
        zeroshot_multitask
        .setInputCols(["sentence"])
        .setOutputCol("extractions")
        .setEntities(entities or [])
        .setEntityThreshold(entity_threshold)
        .setStructures(structures or [])
        .setStructureThreshold(structure_threshold)
        .setClassifications(classifications or [])
        .setClassificationThreshold(classification_threshold)
        .setRelations(relations or [])
        .setRelationThreshold(relation_threshold)
    )

    pipeline = Pipeline(
        stages=[
            document_assembler, 
            sentence_detector,
            #tokenizer,
            #document_splitter,
            zero_shot
    ])
    return pipeline.fit(spark.createDataFrame([[""]]).toDF("text"))

In [18]:
pipe_model = build_multitask_pipeline(**zeroshot_input_prompt_dict)

## 4. Run on data

In [ ]:
sample_texts = [
    """The patient is a 21-day-old Caucasian male here for 2 days of congestion - mom has been suctioning yellow discharge from the patient's nares, plus she has noticed some mild problems with his breathing while feeding (but negative for any perioral cyanosis or retractions). One day ago, mom also noticed a tactile temperature and gave the patient Tylenol. Baby-girl also has had some decreased p.o. intake. His normal breast-feeding is down from 20 minutes q.2h. to 5 to 10 minutes secondary to his respiratory congestion. He sleeps well, but has been more tired and has been fussy over the past 2 days. The parents noticed no improvement with albuterol treatments given in the ER. His urine output has also decreased; normally he has 8 to 10 wet and 5 dirty diapers per 24 hours, now he has down to 4 wet diapers per 24 hours. Mom denies any diarrhea. His bowel movements are yellow colored and soft in nature.""",
    """Progress Note: Jennifer Smith is a 58-year-old woman with type 2 diabetes mellitus and hypertension. She was started on metformin 500mg oral twice daily to control her blood sugar. An HbA1c test was ordered to assess glycemic control. Discharge Summary: The patient underwent an appendectomy for acute appendicitis. Postoperatively he received ibuprofen 400mg tablet orally every 6 hours for pain. A complete blood count was performed to rule out infection.""",
    """He was given boluses of MS04 with some effect, he has since been placed on a PCA - he take 80mg of oxycontin at home, his PCA dose is ~ 2 the morphine dose of the oxycontin, he has also received ativan for anxiety. Repleted with 20 meq kcl po, 30 mmol K-phos iv and 2 gms mag so4 iv. Size: Prostate gland measures 10x1.1x4.9 cm (LS x AP x TS). Estimated volume is 51.9 ml  and is mildly enlarged in size. Normal delineation pattern of the prostate gland is preserved."""
]

data = spark.createDataFrame([(t,) for t in sample_texts]).toDF("text")
results = pipe_model.transform(data).cache()
# Carry `sentence` alongside `extractions`: rows.to_service_rows() turns each annotation's
# metadata["sentence"] index into the real sentence text using this column.
collected = results.select("sentence", "extractions").collect()

# quick peek: all four task types land in one column as chunk / category / struct
results.selectExpr("explode(extractions) as e") \
       .selectExpr("e.annotatorType as type", "e.result as result",
                   "e.begin as begin", "e.end as end",
                   "e.metadata as metadata") \
       .show(60, truncate=False)

In [20]:
results.selectExpr("explode(sentence) as splits") \
    .show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|splits                                                                                                                                                                                                                                                                                                                  |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{document, 0, 270, The patient is a 21-day-old Caucasi

## 5. Flattened JSON output

Split the single `extractions` column into everything each annotation type carries:
- **chunk** → entity: `result, begin, end, entity, confidence`
- **struct** → one record per field: `struct_name, entity (field), result, begin, end, confidence`
- **category / classification** → `task, result, confidence`
- **category / relation** → `result, entity1, chunk1, entity1_begin, entity1_end, chunk1_confidence, entity2, chunk2, entity2_begin, entity2_end, chunk2_confidence`


In [21]:
def _int(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        return x

def _flt(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return x

def _parse_field(raw):
    try:
        return json.loads(raw)
    except (TypeError, ValueError):
        return None

def assemble_json(extractions):
    """Split one document's `extractions` into everything each annotation type carries."""
    out = {"entities": [], "structures": [], "classifications": [], "relations": []}

    for e in extractions:
        atype, m = e.annotatorType, (e.metadata or {})

        if atype == "chunk":                                        # NER
            out["entities"].append({
                "annotatorType": atype, "result": e.result,
                "begin": e.begin, "end": e.end,
                "entity": m.get("entity"), "confidence": _flt(m.get("confidence")),
            })

        elif atype == "struct":                                     # structured extraction
            for field, raw in m.items():                            # one record per field
                f = _parse_field(raw)
                if not isinstance(f, dict) or "text" not in f:      # skip meta (sentence, instance_idx)
                    continue
                out["structures"].append({
                    "annotatorType": atype, "struct_name": e.result,
                    "entity": field, "result": f.get("text"),
                    "begin": f.get("start"), "end": f.get("end"),
                    "confidence": f.get("confidence"),
                    "instance_idx": _int(m.get("instance_idx")),
                })

        elif atype == "category":
            if m.get("category_type") == "relation" or "chunk1" in m:   # relation
                out["relations"].append({
                    "annotatorType": atype, "category_type": "relation",
                    "result": e.result,
                    "entity1": m.get("entity1"), "chunk1": m.get("chunk1"),
                    "entity1_begin": _int(m.get("entity1_begin")), "entity1_end": _int(m.get("entity1_end")),
                    "chunk1_confidence": _flt(m.get("chunk1_confidence")),
                    "entity2": m.get("entity2"), "chunk2": m.get("chunk2"),
                    "entity2_begin": _int(m.get("entity2_begin")), "entity2_end": _int(m.get("entity2_end")),
                    "chunk2_confidence": _flt(m.get("chunk2_confidence")),
                })
            else:                                                       # classification
                out["classifications"].append({
                    "annotatorType": atype, "category_type": m.get("category_type"),
                    "task": m.get("task"), "result": e.result,
                    "begin": e.begin, "end": e.end, "confidence": _flt(m.get("confidence")),
                })

    return out

for i, row in enumerate(collected):
    print(f"===== DOCUMENT {i} =====")
    print(json.dumps(assemble_json(row.extractions), indent=2))


===== DOCUMENT 0 =====
{
  "entities": [
    {
      "annotatorType": "chunk",
      "result": "congestion",
      "begin": 62,
      "end": 71,
      "entity": "PROBLEM",
      "confidence": 0.89518344
    },
    {
      "annotatorType": "chunk",
      "result": "tactile temperature",
      "begin": 304,
      "end": 322,
      "entity": "PROBLEM",
      "confidence": 0.9990017
    },
    {
      "annotatorType": "chunk",
      "result": "Tylenol",
      "begin": 345,
      "end": 351,
      "entity": "MEDICATION",
      "confidence": 0.9986514
    },
    {
      "annotatorType": "chunk",
      "result": "decreased p.o. intake",
      "begin": 382,
      "end": 402,
      "entity": "PROBLEM",
      "confidence": 0.9681797
    },
    {
      "annotatorType": "chunk",
      "result": "breast-feeding",
      "begin": 416,
      "end": 429,
      "entity": "PROCEDURE",
      "confidence": 0.9147765
    },
    {
      "annotatorType": "chunk",
      "result": "respiratory congestion",
    

## 6. custom_nlp_services output

Re-shape the same `extractions` into the **flat row-per-annotation** format that
`custom_nlp_service` writes to the `custom_nlp_results` index (and returns from
`GET /api/v1/results/{session_id}`) — so notebook output can be compared 1:1 with the service.

Mirrors `_collect_multitask_rows()` + `run()` in `custom_nlp_service/app/runners.py`. The notebook
can't import the service package, so the logic is duplicated here — **keep the two in sync** (or
import a shared module once one exists).

| annotation | rows | `label` | `span_text` | `start` / `end` | `confidence` |
|---|---|---|---|---|---|
| `chunk` (ner) | 1 | `metadata.entity` | chunk text | annotation begin/end | `metadata.confidence` |
| `category` **with** `task` (classification) | 1 | `metadata.task` | predicted label | annotation begin/end | `metadata.confidence` |
| `category` **without** `task` (relation) | **2** — head + tail | relation name | `chunk1` / `chunk2` | that entity's offsets | `chunk1/2_confidence` |
| `struct` | **1 per field** | field name | field `text` | field `start` / `end` | field `confidence` |

Relation rows carry `raw_metadata.role` (`head`/`tail`) and share a `relation_id`; structure rows
share a `structure_id` + `instance_idx` — so a full relation or structure instance can be regrouped.

`row_id` / `document_id` / `session_id` / `result_id` are `None` here: the notebook has no
Elasticsearch source document and no service session.


In [ ]:
# This cell used to duplicate custom_nlp_service's _collect_multitask_rows() by hand, with a note
# to import a shared module "once one exists". It exists now: nlp-services-standalone/app/rows.py.
# The standalone service calls the same function, so notebook rows and service rows are produced
# by one implementation rather than two copies -- run tools/parity_check.py to diff them.
import json
import sys

sys.path.insert(0, "/home/ubuntu/cabir/multitask-gliner2/multitask-container/nlp-services-standalone/app")

from rows import to_service_rows

# Keep in sync with the model loaded in section 3. (This previously said
# "zeroshot_multitask_generic" while section 3 loaded the base model -- rows were tagged with a
# pipeline_id the notebook had not actually run.)
MODEL_ID = "zeroshot_multitask_base"
SOURCE_INDEX = "raw_extractions"
JOB_DETAILS = json.dumps({"source": "jupyter", "mode": "zero_shot_multitask"})

# document_id is "doc-{i}" rather than None because rows.py hashes it into relation_id /
# structure_id; sending the service the same ids makes even those hashes comparable.
# `sentences=collected_row.sentence` is what lets each row carry real sentence TEXT (with the
# integer index kept as raw_metadata.sentence_number).
service_rows = [
    row
    for i, collected_row in enumerate(collected)
    for row in to_service_rows(
        collected_row.extractions,
        pipeline_id=MODEL_ID,
        source_index=SOURCE_INDEX,
        common={
            "row_id": None,
            "document_id": f"doc-{i}",
            "patient_id": None,
            "visit_id": None,
            "job_id": None,
            "batch_id": None,
        },
        sentences=collected_row.sentence,
        job_details=JOB_DETAILS,
    )
]

print(f"{len(service_rows)} service rows")
print(json.dumps(service_rows, indent=2))

# For tools/parity_check.py --expect
with open("/tmp/notebook_rows.json", "w") as f:
    json.dump(service_rows, f, indent=2)
print("wrote /tmp/notebook_rows.json")

In [15]:
df = pd.DataFrame(service_rows)
df["task_type"] = df["raw_metadata"].apply(lambda m: m.get("task_type"))

display(df[["pipeline_id", "task_type", "sentence", "span_text",
            "start", "end", "label", "confidence", "engine"]])

df["task_type"].value_counts()

,pipeline_id,task_type,sentence,span_text,start,end,label,confidence,engine
0,zeroshot_multitask_generic,ner,0,congestion,62,71,PROBLEM,0.895183,jsl_zero_shot_multitask
1,zeroshot_multitask_generic,classification,0,Discharge Summary,0,270,document_type,0.584378,jsl_zero_shot_multitask
2,zeroshot_multitask_generic,ner,1,tactile temperature,304,322,PROBLEM,0.999002,jsl_zero_shot_multitask
3,zeroshot_multitask_generic,ner,1,Tylenol,345,351,MEDICATION,0.998651,jsl_zero_shot_multitask
4,zeroshot_multitask_generic,classification,1,Progress Note,272,352,document_type,0.757954,jsl_zero_shot_multitask
...,...,...,...,...,...,...,...,...,...
77,zeroshot_multitask_generic,ner,2,Prostate gland,290,303,PROBLEM,0.756746,jsl_zero_shot_multitask
78,zeroshot_multitask_generic,classification,2,Radiology Report,284,342,document_type,0.996204,jsl_zero_shot_multitask
79,zeroshot_multitask_generic,classification,3,Progress Note,344,403,document_type,0.848989,jsl_zero_shot_multitask
80,zeroshot_multitask_generic,ner,4,prostate gland,439,452,PROBLEM,0.627685,jsl_zero_shot_multitask


task_type
ner               32
classification    20
structure         18
relation          12
Name: count, dtype: int64